# ⚡ Power Grid AI Analytics
### AI-Driven Load Forecasting & Anomaly Detection for Power Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

---

**Topics covered:**
- 📊 Exploratory Data Analysis on PJM hourly load data
- 🔮 Short-Term Load Forecasting: Linear Regression, Random Forest, XGBoost, LSTM
- 🚨 Anomaly Detection: Isolation Forest + LSTM Autoencoder

**Dataset:** PJM AEP Region Hourly Energy Consumption (synthetic replica with real statistical properties)

> To use the **real** PJM dataset, download `AEP_hourly.csv` from [Kaggle](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption), upload it to Colab, and skip the synthetic data cell.

## 0. Install & Import Dependencies

In [ ]:
!pip install xgboost --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras import layers, models
tf.get_logger().setLevel('ERROR')

print('All imports successful')
print(f'  TensorFlow : {tf.__version__}')
print(f'  XGBoost    : {xgb.__version__}')

## 1. Plot Style

In [ ]:
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#3a3f55',
    'axes.labelcolor':  '#c8cad8',
    'xtick.color':      '#8a8fa8',
    'ytick.color':      '#8a8fa8',
    'text.color':       '#e0e2ee',
    'grid.color':       '#2a2f45',
    'grid.linewidth':   0.6,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'legend.framealpha': 0.3,
    'legend.facecolor': '#1a1d27',
    'legend.edgecolor': '#3a3f55',
})

ACCENT  = '#4f8ef7'
ACCENT2 = '#f7834f'
ACCENT3 = '#4ff7a8'
WARN    = '#f74f4f'
print('Plot style configured')

## 2. Data Generation

Generates a realistic PJM AEP-region synthetic dataset capturing:
- Long-term growth trend
- Annual & weekly seasonality
- Daily demand profile (morning ramp, evening peak)
- Stochastic noise
- Injected anomalies (simulating large-load spikes from AI data centers)

In [ ]:
def generate_pjm_data(start='2017-01-01', end='2023-12-31', seed=42, anomaly_fraction=0.003):
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start=start, end=end, freq='h')
    n   = len(idx)
    hour_of_day = idx.hour.values
    day_of_week = idx.dayofweek.values
    day_of_year = idx.dayofyear.values
    year_frac   = ((idx - pd.Timestamp(start)).total_seconds() / (365.25 * 24 * 3600)).values
    base    = 13500 + 200 * year_frac
    annual  = (2200 * np.sin(2 * np.pi * (day_of_year - 172) / 365) +
               600  * np.sin(4 * np.pi * (day_of_year - 355) / 365))
    daily   = (1800 * np.exp(-((hour_of_day - 8)  ** 2) / 8) +
               2000 * np.exp(-((hour_of_day - 19) ** 2) / 6) -
               800  * np.exp(-((hour_of_day - 3)  ** 2) / 4))
    weekend = -1500 * (day_of_week >= 5).astype(float)
    noise   = rng.normal(0, 400, n)
    load    = np.clip(base + annual + daily + weekend + noise, 8000, 24000)
    n_anom   = max(1, int(n * anomaly_fraction))
    anom_idx = rng.choice(n, n_anom, replace=False)
    anom_mag = rng.choice([-1, 1], n_anom) * rng.uniform(3500, 6000, n_anom)
    load_arr = load.copy()
    load_arr[anom_idx] += anom_mag
    load = np.clip(load_arr, 5000, 30000)
    labels = np.zeros(n, dtype=int)
    labels[anom_idx] = 1
    df = pd.DataFrame({'datetime': idx, 'load_mw': load.round(1), 'is_anomaly': labels})
    return df.set_index('datetime')

def add_features(df):
    df = df.copy()
    df['hour']       = df.index.hour
    df['dayofweek']  = df.index.dayofweek
    df['month']      = df.index.month
    df['dayofyear']  = df.index.dayofyear
    df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
    df['quarter']    = df.index.quarter
    for lag in [1, 2, 24, 168]:
        df[f'lag_{lag}h'] = df['load_mw'].shift(lag)
    df['roll_24h_mean'] = df['load_mw'].shift(1).rolling(24).mean()
    df['roll_24h_std']  = df['load_mw'].shift(1).rolling(24).std()
    return df.dropna()

df      = generate_pjm_data()
df_feat = add_features(df)
print(f'Dataset ready: {len(df):,} hourly records | {df["is_anomaly"].sum()} injected anomalies')
df.describe().round(1)

## 3. Exploratory Data Analysis

### 3.1 Full Time Series + Seasonal Profiles

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), facecolor='#0f1117')
fig.suptitle('PJM AEP Region - Hourly Load Overview', fontsize=15, fontweight='bold', y=0.98)

ax = axes[0]
daily = df['load_mw'].resample('D').mean()
ax.fill_between(daily.index, daily.values, alpha=0.3, color=ACCENT)
ax.plot(daily.index, daily.values, color=ACCENT, linewidth=0.8, label='Daily avg load')
ax.set_ylabel('Load (MW)'); ax.set_title('Full Time Series (Daily Average)')
ax.legend(); ax.grid(True)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax = axes[1]
monthly = [df.loc[df.index.month == m, 'load_mw'].values for m in range(1, 13)]
bp = ax.boxplot(monthly, patch_artist=True, medianprops=dict(color=ACCENT3, linewidth=2))
for patch in bp['boxes']:
    patch.set_facecolor('#2a2f45'); patch.set_alpha(0.8)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Load (MW)'); ax.set_title('Monthly Load Distribution'); ax.grid(True, axis='y')

ax = axes[2]
seasons = {'Winter':[12,1,2],'Spring':[3,4,5],'Summer':[6,7,8],'Fall':[9,10,11]}
clrs    = [ACCENT, ACCENT3, WARN, ACCENT2]
for (season, months), color in zip(seasons.items(), clrs):
    mask = df.index.month.isin(months)
    profile = df.loc[mask].groupby(df.loc[mask].index.hour)['load_mw'].mean()
    ax.plot(profile.index, profile.values, color=color, linewidth=2.2, label=season)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Avg Load (MW)')
ax.set_title('Average Daily Load Profile by Season')
ax.set_xticks(range(0, 24, 2)); ax.legend(); ax.grid(True)
fig.tight_layout(); plt.show()

### 3.2 Load Heatmap (Hour x Day-of-Week)

In [ ]:
pivot = df['load_mw'].groupby([df.index.hour, df.index.dayofweek]).mean().unstack()
pivot.columns = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
fig, ax = plt.subplots(figsize=(10, 7), facecolor='#0f1117')
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlBu_r', origin='lower')
ax.set_xticks(range(7)); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(0, 24, 2))
ax.set_yticklabels([f'{h:02d}:00' for h in range(0, 24, 2)])
ax.set_xlabel('Day of Week'); ax.set_ylabel('Hour of Day')
ax.set_title('Average Load Heatmap (Hour x Day-of-Week)', fontweight='bold')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label('Avg Load (MW)', color='#c8cad8')
fig.tight_layout(); plt.show()

## 4. Short-Term Load Forecasting

Using a **temporal train/test split** (last 6 months = test) to prevent data leakage.

In [ ]:
FEATURE_COLS = [
    'hour','dayofweek','month','dayofyear','is_weekend','quarter',
    'lag_1h','lag_2h','lag_24h','lag_168h','roll_24h_mean','roll_24h_std'
]

def temporal_split(df, test_months=6):
    cutoff = df.index[-1] - pd.DateOffset(months=test_months)
    return df[df.index <= cutoff], df[df.index > cutoff]

def show_metrics(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'  {name:<22}  MAE={mae:7.1f} MW   RMSE={rmse:7.1f} MW   MAPE={mape:.2f}%')
    return {'Model':name,'MAE (MW)':f'{mae:.1f}','RMSE (MW)':f'{rmse:.1f}','MAPE (%)':f'{mape:.2f}'}

train, test = temporal_split(df_feat)
X_train, y_train = train[FEATURE_COLS].values, train['load_mw'].values
X_test,  y_test  = test[FEATURE_COLS].values,  test['load_mw'].values
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
results   = {}
all_metrics = []
print(f'Train: {len(train):,} hours | Test: {len(test):,} hours')
print(f'Test period: {test.index[0].date()} to {test.index[-1].date()}')

### 4.1 Linear Regression (Baseline)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s)
results['Linear Regression'] = pred_lr
all_metrics.append(show_metrics('Linear Regression', y_test, pred_lr))

### 4.2 Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=150, max_depth=12, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
results['Random Forest'] = pred_rf
all_metrics.append(show_metrics('Random Forest', y_test, pred_rf))

### 4.3 XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0, n_jobs=-1
)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
pred_xgb = xgb_model.predict(X_test)
results['XGBoost'] = pred_xgb
all_metrics.append(show_metrics('XGBoost', y_test, pred_xgb))

### 4.4 LSTM

In [ ]:
SEQ_LEN = 24
sc2     = StandardScaler()
tr_vals = sc2.fit_transform(train['load_mw'].values.reshape(-1,1)).flatten()
te_vals = sc2.transform(test['load_mw'].values.reshape(-1,1)).flatten()

def make_sequences(arr, sl):
    X, y = [], []
    for i in range(len(arr) - sl):
        X.append(arr[i:i+sl]); y.append(arr[i+sl])
    return np.array(X)[..., np.newaxis], np.array(y)

X_tr, y_tr = make_sequences(tr_vals, SEQ_LEN)

lstm_model = models.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

history = lstm_model.fit(X_tr, y_tr, epochs=15, batch_size=256, validation_split=0.1, verbose=1)

In [ ]:
buffer = list(tr_vals[-SEQ_LEN:])
preds  = []
for true_val in te_vals:
    x_in = np.array(buffer[-SEQ_LEN:])[np.newaxis, :, np.newaxis]
    preds.append(lstm_model.predict(x_in, verbose=0)[0, 0])
    buffer.append(true_val)

pred_lstm = sc2.inverse_transform(np.array(preds).reshape(-1,1)).flatten()
results['LSTM'] = pred_lstm
all_metrics.append(show_metrics('LSTM', y_test, pred_lstm))

fig, ax = plt.subplots(figsize=(8,4), facecolor='#0f1117')
ax.plot(history.history['loss'],     color=ACCENT,  label='Train Loss')
ax.plot(history.history['val_loss'], color=ACCENT2, label='Val Loss', linestyle='--')
ax.set_title('LSTM Training Loss', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

### 4.5 Forecast Comparison

In [ ]:
window     = slice(0, 168)
colors_map = {'Linear Regression':ACCENT,'Random Forest':ACCENT2,'XGBoost':ACCENT3,'LSTM':WARN}

fig = plt.figure(figsize=(14, 13), facecolor='#0f1117')
gs  = gridspec.GridSpec(len(results)+1, 1, hspace=0.45)

ax0 = fig.add_subplot(gs[0])
ax0.plot(test.index[window], y_test[window], color='white', linewidth=2, label='Actual', zorder=5)
for name, pred in results.items():
    ax0.plot(test.index[window], pred[window], color=colors_map[name],
             linewidth=1.4, alpha=0.85, linestyle='--', label=name)
ax0.set_title('Forecast vs Actual - First Week of Test Set', fontweight='bold')
ax0.set_ylabel('Load (MW)'); ax0.legend(ncol=2); ax0.grid(True)

for i, (name, pred) in enumerate(results.items()):
    ax = fig.add_subplot(gs[i+1])
    residuals = y_test[window] - pred[window]
    ax.bar(range(len(residuals)), residuals, color=colors_map[name], alpha=0.7, width=1.0)
    ax.axhline(0, color='white', linewidth=0.8)
    ax.set_title(f'{name} - Residuals'); ax.set_ylabel('Error (MW)'); ax.grid(True, axis='y')

plt.show()

### 4.6 Model Performance Summary

In [ ]:
metrics_df = pd.DataFrame(all_metrics).set_index('Model')

fig, ax = plt.subplots(figsize=(9, 3), facecolor='#0f1117')
ax.axis('off')
tbl = ax.table(cellText=metrics_df.values, colLabels=metrics_df.columns,
               rowLabels=metrics_df.index, cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.3, 2.2)
for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor('#0f1117'); cell.set_text_props(color='#e0e2ee'); cell.set_edgecolor('#3a3f55')
    if r == 0: cell.set_facecolor('#2a3a5e'); cell.set_text_props(color=ACCENT3, fontweight='bold')
    if c == -1: cell.set_facecolor('#1a2a3e'); cell.set_text_props(color=ACCENT2, fontweight='bold')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout(); plt.show()
print(metrics_df.to_string())

## 5. Anomaly Detection

Both models are **fully unsupervised** - no anomaly labels used during training.

### 5.1 Isolation Forest

In [ ]:
feats  = ['load_mw','hour','dayofweek','month']
X_ad_s = StandardScaler().fit_transform(df_feat[feats].values)
iso    = IsolationForest(contamination=0.003, random_state=42, n_jobs=-1)
df_iso = df_feat.copy()
df_iso['iso_pred'] = (iso.fit_predict(X_ad_s) == -1).astype(int)

tp = int(((df_iso['iso_pred']==1)&(df_iso['is_anomaly']==1)).sum())
fp = int(((df_iso['iso_pred']==1)&(df_iso['is_anomaly']==0)).sum())
fn = int(((df_iso['iso_pred']==0)&(df_iso['is_anomaly']==1)).sum())
prec=tp/(tp+fp+1e-9); rec=tp/(tp+fn+1e-9); f1=2*prec*rec/(prec+rec+1e-9)
print(f'Isolation Forest  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}')

In [ ]:
sample = df_iso.iloc[-24*60:]
fig, axes = plt.subplots(2,1,figsize=(14,8),facecolor='#0f1117')
fig.suptitle('Isolation Forest - Anomaly Detection',fontsize=14,fontweight='bold')

ax = axes[0]
ax.plot(sample.index, sample['load_mw'], color='#c8cad8', linewidth=0.7, label='Load')
det = sample[sample['iso_pred']==1]
ax.scatter(det.index, det['load_mw'], color=WARN, s=40, zorder=5,
           label='Detected', edgecolors='white', linewidths=0.4)
tp_s = sample[(sample['iso_pred']==1)&(sample['is_anomaly']==1)]
ax.scatter(tp_s.index, tp_s['load_mw'], color=ACCENT3, s=60, zorder=6, label='True positive', marker='*')
ax.set_ylabel('Load (MW)'); ax.legend(); ax.grid(True)

ax2 = axes[1]
tn = int(((sample['iso_pred']==0)&(sample['is_anomaly']==0)).sum())
bars=ax2.bar(['True Pos','False Pos','False Neg','True Neg'],[tp,fp,fn,tn],
             color=[ACCENT3,WARN,ACCENT2,ACCENT],alpha=0.85,edgecolor='#3a3f55')
for bar,val in zip(bars,[tp,fp,fn,tn]):
    ax2.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,str(val),
             ha='center',va='bottom',fontsize=12,fontweight='bold')
ax2.set_ylabel('Count'); ax2.grid(True,axis='y')
fig.tight_layout(); plt.show()

### 5.2 LSTM Autoencoder

In [ ]:
SEQ_AE  = 24
sc_ae   = StandardScaler()
vals_ae = sc_ae.fit_transform(df_feat['load_mw'].values.reshape(-1,1)).flatten()
X_ae    = np.array([vals_ae[i:i+SEQ_AE] for i in range(len(vals_ae)-SEQ_AE)])[...,np.newaxis]
n_tr_ae = int(len(X_ae)*0.8)

ae = models.Sequential([
    layers.LSTM(32, return_sequences=False, input_shape=(SEQ_AE,1), name='encoder'),
    layers.RepeatVector(SEQ_AE),
    layers.LSTM(32, return_sequences=True, name='decoder'),
    layers.TimeDistributed(layers.Dense(1))
])
ae.compile(optimizer='adam', loss='mse')
ae.summary()
ae.fit(X_ae[:n_tr_ae], X_ae[:n_tr_ae], epochs=10, batch_size=512, validation_split=0.1, verbose=1)

In [ ]:
X_rec     = ae.predict(X_ae, verbose=0)
errors    = np.mean((X_ae - X_rec)**2, axis=(1,2))
threshold = np.percentile(errors[:n_tr_ae], 99.7)
print(f'Threshold (99.7th pct on train): {threshold:.5f}')

fig, ax = plt.subplots(figsize=(14,4), facecolor='#0f1117')
ax.plot(errors, color=ACCENT, linewidth=0.6, alpha=0.8, label='Reconstruction error')
ax.axhline(threshold, color=WARN, linewidth=1.8, linestyle='--', label=f'Threshold={threshold:.4f}')
ax.fill_between(range(len(errors)), errors, threshold,
                where=(errors>threshold), color=WARN, alpha=0.25, label='Anomaly region')
ax.set_xlabel('Sample'); ax.set_ylabel('MSE')
ax.set_title('LSTM Autoencoder - Reconstruction Error', fontweight='bold')
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
padded = np.zeros(len(df_feat), dtype=int)
padded[SEQ_AE:SEQ_AE+len(errors)] = (errors > threshold).astype(int)
df_ae = df_feat.copy(); df_ae['ae_pred'] = padded

tp2=int(((df_ae['ae_pred']==1)&(df_ae['is_anomaly']==1)).sum())
fp2=int(((df_ae['ae_pred']==1)&(df_ae['is_anomaly']==0)).sum())
fn2=int(((df_ae['ae_pred']==0)&(df_ae['is_anomaly']==1)).sum())
prec2=tp2/(tp2+fp2+1e-9); rec2=tp2/(tp2+fn2+1e-9); f12=2*prec2*rec2/(prec2+rec2+1e-9)
print(f'LSTM Autoencoder  Precision={prec2:.3f}  Recall={rec2:.3f}  F1={f12:.3f}')

s2=df_ae.iloc[-24*60:]
fig,axes=plt.subplots(2,1,figsize=(14,8),facecolor='#0f1117')
fig.suptitle('LSTM Autoencoder - Anomaly Detection',fontsize=14,fontweight='bold')
ax=axes[0]
ax.plot(s2.index,s2['load_mw'],color='#c8cad8',linewidth=0.7,label='Load')
d2=s2[s2['ae_pred']==1]
ax.scatter(d2.index,d2['load_mw'],color=WARN,s=40,zorder=5,label='Detected',edgecolors='white',linewidths=0.4)
tp_s2=s2[(s2['ae_pred']==1)&(s2['is_anomaly']==1)]
ax.scatter(tp_s2.index,tp_s2['load_mw'],color=ACCENT3,s=60,zorder=6,label='True positive',marker='*')
ax.set_ylabel('Load (MW)'); ax.legend(); ax.grid(True)
ax2=axes[1]
tn2=int(((s2['ae_pred']==0)&(s2['is_anomaly']==0)).sum())
bars2=ax2.bar(['True Pos','False Pos','False Neg','True Neg'],[tp2,fp2,fn2,tn2],
              color=[ACCENT3,WARN,ACCENT2,ACCENT],alpha=0.85,edgecolor='#3a3f55')
for b,v in zip(bars2,[tp2,fp2,fn2,tn2]):
    ax2.text(b.get_x()+b.get_width()/2,b.get_height()+0.3,str(v),
             ha='center',va='bottom',fontsize=12,fontweight='bold')
ax2.set_ylabel('Count'); ax2.grid(True,axis='y')
fig.tight_layout(); plt.show()

## 6. Final Summary

In [ ]:
print('=' * 52)
print('  FORECASTING RESULTS')
print('=' * 52)
print(metrics_df.to_string())

print('\n' + '=' * 52)
print('  ANOMALY DETECTION RESULTS')
print('=' * 52)
ad_df = pd.DataFrame({
    'Precision': [f'{prec:.3f}',  f'{prec2:.3f}'],
    'Recall':    [f'{rec:.3f}',   f'{rec2:.3f}'],
    'F1':        [f'{f1:.3f}',    f'{f12:.3f}'],
}, index=['Isolation Forest', 'LSTM Autoencoder'])
print(ad_df.to_string())
print('\nDone!')

---

## Future Directions

- Graph Neural Networks (GNN) for multi-bus topology-aware forecasting
- Temporal Fusion Transformer (TFT) for probabilistic multi-step forecasting
- AI data center load profiling - characterizing GPU cluster ramp signatures
- Online anomaly detection for streaming SCADA telemetry
- Integration with OpenDSS / MATPOWER for physics-informed ML

---
*Built as part of research preparation in AI for power systems analytics.*